In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from time import sleep

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 6)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("No GitHub tokens found in All_Tokens.env")

token_index = 0
def get_headers():
    global token_index
    print(f"🔁 Using token #{token_index + 1}")
    token = tokens[token_index]
    token_index = (token_index + 1) % len(tokens)
    return {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-repo-crawler/1.0"
    }

# === File paths ===
input_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step1_search_output.csv"
output_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step2_verified_output.csv"
failed_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step2_failed_requests.csv"

# === Read input ===
df = pd.read_csv(input_path)

statuses = []
reasons = []
failed_repos = []

MAX_RETRIES = 3

# === Verify each repo using REST API ===
for i, row in df.iterrows():
    repo = row["full_name"]
    url = f"https://api.github.com/repos/{repo}"

    retries = 0
    while retries < MAX_RETRIES:
        response = requests.get(url, headers=get_headers())
        if response.status_code == 403:
            print(f"⏳ 403 Rate limit or abuse protection triggered for {repo}. Retrying in 10s...")
            retries += 1
            sleep(10)
            continue
        break

    if response.status_code != 200:
        print(f"❌ Error on {repo}: HTTP {response.status_code}")
        statuses.append("reject")
        reasons.append(f"HTTP {response.status_code}")
        failed_repos.append({"full_name": repo, "error_code": response.status_code})
        sleep(1)
        continue

    data = response.json()
    if data.get("fork", True):
        statuses.append("reject")
        reasons.append("fork")
    elif data.get("archived", True):
        statuses.append("reject")
        reasons.append("archived")
    elif data.get("stargazers_count", 0) <= 50:
        statuses.append("reject")
        reasons.append("low stars")
    elif data.get("language") not in ["Java", "Kotlin", "Dart"]:
        statuses.append("reject")
        reasons.append("language mismatch")
    else:
        statuses.append("pass")
        reasons.append("")

    if i % 100 == 0:
        print(f"✅ Verified {i+1} repos...")

# === Save outputs ===
df["status"] = statuses
df["reason"] = reasons
df.to_csv(output_path, index=False)
print(f"✅ Step 2 complete. Verified data saved to: {output_path}")

if failed_repos:
    pd.DataFrame(failed_repos).to_csv(failed_path, index=False)
    print(f"⚠️ Failed requests saved to: {failed_path}")
